In [15]:
import pygame
import random

# Initialize pygame
pygame.init()

# Game Constants
WIDTH, HEIGHT = 800, 400
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
GROUND_Y = HEIGHT - 50

# Load assets
dino_img = pygame.image.load("dino.png")
dino_img = pygame.transform.scale(dino_img, (50, 50))
dino_crouch_img = pygame.transform.scale(dino_img, (50, 30))
cactus_img = pygame.image.load("cactus.png")
cactus_img = pygame.transform.scale(cactus_img, (30, 50))

# Set up screen
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Dino Game")
clock = pygame.time.Clock()

# Font for replay button and score
font = pygame.font.Font(None, 36)

def draw_replay_button():
    text = font.render("Replay", True, BLACK)
    rect = text.get_rect(center=(WIDTH//2, HEIGHT//2))
    pygame.draw.rect(screen, WHITE, rect.inflate(20, 10))
    screen.blit(text, rect)
    return rect
D  
# Dinosaur class
class Dinosaur:
    def __init__(self):
        self.x = 50
        self.y = GROUND_Y - 50
        self.vel_y = 0
        self.gravity = 1
        self.is_jumping = False
        self.is_crouching = False

    def jump(self):
        if not self.is_jumping and not self.is_crouching:
            self.vel_y = -15
            self.is_jumping = True

    def crouch(self, state):
        self.is_crouching = state
        if state:
            self.y = GROUND_Y - 30
        else:
            self.y = GROUND_Y - 50

    def update(self):
        if not self.is_crouching:
            self.y += self.vel_y
            self.vel_y += self.gravity
            if self.y >= GROUND_Y - 50:
                self.y = GROUND_Y - 50
                self.is_jumping = False

    def draw(self):
        if self.is_crouching:
            screen.blit(dino_crouch_img, (self.x, self.y))
        else:
            screen.blit(dino_img, (self.x, self.y))

# Obstacle class
class Obstacle:
    def __init__(self, speed):
        self.x = WIDTH
        self.y = GROUND_Y - 50
        self.speed = speed

    def update(self):
        self.x -= self.speed
        if self.x < -30:
            self.x = WIDTH + random.randint(200, 400)

    def draw(self):
        screen.blit(cactus_img, (self.x, self.y))

# Game loop
def game_loop():
    dino = Dinosaur()
    score = 0
    speed = 10
    obstacles = [Obstacle(speed)]
    running = True
    game_over = False
    
    while running:
        screen.fill(WHITE)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE and not game_over:
                    dino.jump()
                if event.key == pygame.K_DOWN:
                    dino.crouch(True)
            if event.type == pygame.KEYUP:
                if event.key == pygame.K_DOWN:
                    dino.crouch(False)
            if game_over and event.type == pygame.MOUSEBUTTONDOWN:
                if replay_rect.collidepoint(event.pos):
                    game_loop()
                    return

        if not game_over:
            dino.update()
            for obstacle in obstacles:
                obstacle.update()
                if dino.x < obstacle.x + 30 and dino.x + 50 > obstacle.x and dino.y + 50 > obstacle.y:
                    game_over = True
                obstacle.draw()

            # Increase score and difficulty
            score += 1
            if score % 500 == 0:
                speed += 1
                obstacles.append(Obstacle(speed))

            dino.draw()
            score_text = font.render(f"Score: {score}", True, BLACK)
            screen.blit(score_text, (10, 10))
        else:
            replay_rect = draw_replay_button()
        
        pygame.display.update()
        clock.tick(30)

    pygame.quit()

game_loop()


In [1]:
import pygame
import random
import json

# Load configuration from a JSON file
with open('config.json') as config_file:
    config = json.load(config_file)

# Initialize pygame
pygame.init()

# Game Constants
WIDTH = config['WIDTH']
HEIGHT = config['HEIGHT']
WHITE = tuple(config['WHITE'])
BLACK = tuple(config['BLACK'])
GROUND_Y = HEIGHT - config['GROUND_OFFSET']
FPS = config['FPS']

# Load assets
dino_img = pygame.image.load(config['DINO_IMAGE'])
dino_img = pygame.transform.scale(dino_img, (50, 50))
dino_crouch_img = pygame.transform.scale(dino_img, (50, 30))
cactus_img = pygame.image.load(config['CACTUS_IMAGE'])
cactus_img = pygame.transform.scale(cactus_img, (30, 50))

# Set up screen
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption(config['GAME_TITLE'])
clock = pygame.time.Clock()

# Font for replay button and score
font = pygame.font.Font(None, 36)

def draw_replay_button():
    text = font.render("Replay", True, BLACK)
    rect = text.get_rect(center=(WIDTH//2, HEIGHT//2))
    pygame.draw.rect(screen, WHITE, rect.inflate(20, 10))
    screen.blit(text, rect)
    return rect

# Dinosaur class
class Dinosaur:
    def __init__(self):
        self.x = 50
        self.y = GROUND_Y - 50
        self.vel_y = 0
        self.gravity = 1
        self.is_jumping = False
        self.is_crouching = False

    def jump(self):
        if not self.is_jumping and not self.is_crouching:
            self.vel_y = -15
            self.is_jumping = True

    def crouch(self, state):
        self.is_crouching = state
        if state:
            self.y = GROUND_Y - 30
        else:
            self.y = GROUND_Y - 50

    def update(self):
        if not self.is_crouching:
            self.y += self.vel_y
            self.vel_y += self.gravity
            if self.y >= GROUND_Y - 50:
                self.y = GROUND_Y - 50
                self.is_jumping = False

    def draw(self):
        if self.is_crouching:
            screen.blit(dino_crouch_img, (self.x, self.y))
        else:
            screen.blit(dino_img, (self.x, self.y))

# Obstacle class
class Obstacle:
    def __init__(self, speed):
        self.x = WIDTH
        self.y = GROUND_Y - 50
        self.speed = speed
        self.type = random.choice(['cactus', 'bird'])  # Add more types as needed

    def update(self):
        self.x -= self.speed
        if self.x < -30:
            self.x = WIDTH + random.randint(200, 400)

    def draw(self):
        screen.blit(cactus_img, (self.x, self.y))

# Game loop
def game_loop():
    dino = Dinosaur()
    score = 0
    speed = 10
    obstacles = [Obstacle(speed)]
    running = True
    game_over = False
    
    while running:
        screen.fill(WHITE)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE and not game_over:
                    dino.jump()
                if event.key == pygame.K_DOWN:
                    dino.crouch(True)
            if event.type == pygame.KEYUP:
                if event.key == pygame.K_DOWN:
                    dino.crouch(False)
            if game_over and event.type == pygame.MOUSEBUTTONDOWN:
                if replay_rect.collidepoint(event.pos):
                    game_loop()
                    return

        if not game_over:
            dino.update()
            for obstacle in obstacles:
                obstacle.update()
                if dino.x < obstacle.x + 30 and dino.x + 50 > obstacle.x and dino.y + 50 > obstacle.y:
                    game_over = True
                obstacle.draw()

            # Increase score and difficulty
            score += 1
            if score % 500 == 0:
                speed += 1
                obstacles.append(Obstacle(speed))

            dino.draw()
            score_text = font.render(f"Score: {score}", True, BLACK)
            screen.blit(score_text, (10, 10))
        else:
            replay_rect = draw_replay_button()
        
        pygame.display.update()
        clock.tick(FPS)

    pygame.quit()

game_loop()


pygame 2.6.1 (SDL 2.28.4, Python 3.12.4)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [5]:
import pygame
import random
import json

# Load configuration from a JSON file
with open('config.json') as config_file:
    config = json.load(config_file)

# Initialize pygame
pygame.init()

# Game Constants
WIDTH = config['WIDTH']
HEIGHT = config['HEIGHT']
WHITE = tuple(config['WHITE'])
BLACK = tuple(config['BLACK'])
GROUND_Y = HEIGHT - config['GROUND_OFFSET']
FPS = config['FPS']

# Load assets
dino_img = pygame.image.load(config['DINO_IMAGE'])
dino_img = pygame.transform.scale(dino_img, (50, 50))
dino_crouch_img = pygame.transform.scale(dino_img, (50, 30))
cactus_img = pygame.image.load(config['CACTUS_IMAGE'])
cactus_img = pygame.transform.scale(cactus_img, (30, 50))
bird_img = pygame.image.load(config['BIRD_IMAGE'])
bird_img = pygame.transform.scale(bird_img, (40, 40))

# Set up screen
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption(config['GAME_TITLE'])
clock = pygame.time.Clock()

# Font for replay button and score
font = pygame.font.Font(None, 36)

# Sound effects (optional, can be removed if not needed)
pygame.mixer.init()
jump_sound = pygame.mixer.Sound(config['JUMP_SOUND'])
collision_sound = pygame.mixer.Sound(config['COLLISION_SOUND'])

# Background
background = pygame.image.load("background.png")
background = pygame.transform.scale(background, (WIDTH, HEIGHT))

# Draw replay button
def draw_replay_button():
    text = font.render("Replay", True, BLACK)
    rect = text.get_rect(center=(WIDTH // 2, HEIGHT // 2))
    pygame.draw.rect(screen, WHITE, rect.inflate(20, 10))
    screen.blit(text, rect)
    return rect

# Dinosaur class
class Dinosaur:
    def __init__(self):
        self.x = 50
        self.y = GROUND_Y - 50
        self.vel_y = 0
        self.gravity = 1
        self.is_jumping = False
        self.is_crouching = False
        self.jump_height = -15

    def jump(self):
        if not self.is_jumping and not self.is_crouching:
            self.vel_y = self.jump_height
            self.is_jumping = True
            jump_sound.play()

    def crouch(self, state):
        self.is_crouching = state
        if state:
            self.y = GROUND_Y - 30
        else:
            self.y = GROUND_Y - 50

    def update(self):
        if not self.is_crouching:
            self.y += self.vel_y
            self.vel_y += self.gravity
            if self.y >= GROUND_Y - 50:
                self.y = GROUND_Y - 50
                self.is_jumping = False

    def draw(self):
        if self.is_crouching:
            screen.blit(dino_crouch_img, (self.x, self.y))
        else:
            screen.blit(dino_img, (self.x, self.y))

# Obstacle class
class Obstacle:
    def __init__(self, speed):
        self.x = WIDTH
        self.y = GROUND_Y - 50
        self.speed = speed
        self.type = random.choice(['cactus', 'bird'])

    def update(self):
        self.x -= self.speed
        if self.x < -30:
            self.x = WIDTH + random.randint(200, 400)
            self.type = random.choice(['cactus', 'bird'])

    def draw(self):
        if self.type == 'cactus':
            screen.blit(cactus_img, (self.x, self.y))
        elif self.type == 'bird':
            screen.blit(bird_img, (self.x, self.y - 30))  # Birds fly slightly above the ground

# Game loop
def game_loop():
    dino = Dinosaur()
    score = 0
    speed = 10
    obstacles = [Obstacle(speed)]
    running = True
    game_over = False

    while running:
        screen.fill(WHITE)
        screen.blit(background, (0, 0))  # Draw the background

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE and not game_over:
                    dino.jump()
                if event.key == pygame.K_DOWN:
                    dino.crouch(True)
            if event.type == pygame.KEYUP:
                if event.key == pygame.K_DOWN:
                    dino.crouch(False)
            if game_over and event.type == pygame.MOUSEBUTTONDOWN:
                if replay_rect.collidepoint(event.pos):
                    game_loop()
                    return

        if not game_over:
            dino.update()
            for obstacle in obstacles:
                obstacle.update()
                if dino.x < obstacle.x + 30 and dino.x + 50 > obstacle.x and dino.y + 50 > obstacle.y:
                    game_over = True
                    collision_sound.play()
                obstacle.draw()

            # Increase score and difficulty
            score += 1
            if score % 500 == 0:
                speed += 1
                obstacles.append(Obstacle(speed))

            dino.draw()
            score_text = font.render(f"Score: {score}", True, BLACK)
            screen.blit(score_text, (10, 10))
        else:
            replay_rect = draw_replay_button()

        pygame.display.update()
        clock.tick(FPS)

    pygame.quit()

game_loop()


In [3]:
import json

config = {
    "WIDTH": 800,
    "HEIGHT": 600,
    "WHITE": [255, 255, 255],
    "BLACK": [0, 0, 0],
    "GROUND_OFFSET": 50,
    "FPS": 60,
    "GAME_TITLE": "Dino Game",
    "DINO_IMAGE": "dino.png",
    "DINO_CROUCH_IMAGE": "dino_crouch.png",
    "CACTUS_IMAGE": "cactus.png",
    "BIRD_IMAGE": "bird.png",
    "BACKGROUND_IMAGE": "background.png",
    "JUMP_SOUND": "jump.wav",
    "COLLISION_SOUND": "collision.wav"
}

# Writing to config.json file
with open('config.json', 'w') as config_file:
    json.dump(config, config_file, indent=4)

print("Config file 'config.json' created successfully.")


Config file 'config.json' created successfully.
